# 03 | Few-shot transfer learning for BHET yield prediction

**JACS Au computational analysis · PES-H1 (PHL7) target enzyme · LCCICCG source enzyme**

This notebook reproduces the three GP regression strategies presented in the transfer-learning section of the manuscript: **target-only**, **full transfer**, and **response-range-calibrated transfer**. Models are evaluated using 8, 10, or 12 target-enzyme training observations over 30 repeated paired splits. Performance is measured as held-out target-enzyme $R^2$.

> **Reproduction note:** The source is `Transfer_learning_yield_R2.ipynb`. Its GP kernel, preprocessing, fitting settings, random splits, and source-data filtering are retained. This cleaned version is not numerically validated until the input workbook is supplied and all cells have run.

## 1. Inputs and computational settings

Place `PET_DegradationTrain_Test.xlsx` in the same directory as this notebook. It must contain sheets `LCC` (440 rows) and `PHL7` (30 rows), with the specified feature and response columns.

The GP uses standardized input variables, a Matérn (ν = 2.5) plus dot-product kernel, and `alpha=0.006`. Hyperparameters are optimized on each training dataset. Fixed random seeds ensure the **same target observations are held out** for the three strategies within each repetition.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, DotProduct
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from threadpoolctl import threadpool_limits

excel_file = Path("PET_DegradationTrain_Test.xlsx")
output_folder = Path("yield_r2_results")
output_folder.mkdir(exist_ok=True)

features = ["family", "time", "substrate", "enzyme", "solvent", "temperature", "ph"]
target = "Mean_BHET"
columns = features + [target]
training_sizes = [8, 10, 12]
n_repeats = 30

def new_model():
    kernel = Matern(nu=2.5, length_scale_bounds=(1e-7, 1e5)) + DotProduct(
        sigma_0=1.0, sigma_0_bounds=(1e-7, 1e5)
    )
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=0.006, normalize_y=True,
        random_state=42, n_restarts_optimizer=0
    )
    return make_pipeline(StandardScaler(), gp)

# Recreating this generator gives exactly the same splits in every model block.
def paired_splits(phl7):
    for n in training_sizes:
        for repeat in range(n_repeats):
            seed = 42 + 100 * n + repeat
            rng = np.random.RandomState(seed)
            train_ids = rng.choice(len(phl7), size=n, replace=False)
            test_ids = np.setdiff1d(np.arange(len(phl7)), train_ids)
            yield n, repeat + 1, seed, train_ids, test_ids

# Store only R², split information, source counts, and the one example plot.
scores = []
calibrated_counts = []
calibrated_membership = []
example_predictions = {}
fit_warnings = []

def evaluate(model_name, training, test, n, repeat, seed, source_count):
    model = new_model()
    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
        warnings.simplefilter("always")
        model.fit(training[features], training[target])
        predicted = model.predict(test[features])

    score = r2_score(test[target], predicted, force_finite=False)
    scores.append({"PHL7 training n": n, "Repeat": repeat, "Seed": seed,
                   "Model": model_name, "LCC training n": source_count,
                   "Test R2": score})
    for warning in caught:
        fit_warnings.append({"Model": model_name, "PHL7 training n": n,
                             "Repeat": repeat, "Warning": str(warning.message)})

    # Fixed example chosen before examining performance: n=10, repeat=1.
    if n == 10 and repeat == 1:
        example_predictions[model_name] = pd.DataFrame({
            "PHL7 row": test.index, "Measured": test[target].to_numpy(),
            "Predicted": predicted
        })

## 2. Target-only GP

Fit one GP using only the 8, 10, or 12 selected PES-H1 observations. Predict BHET yield for the remaining PES-H1 observations. This provides the small-data baseline.

In [ ]:
phl7 = pd.read_excel(excel_file, sheet_name="PHL7", usecols=columns)
assert len(phl7) == 30
assert np.isfinite(phl7[columns].to_numpy()).all()
assert not phl7.duplicated(features).any()

for n, repeat, seed, train_ids, test_ids in paired_splits(phl7):
    training = phl7.iloc[train_ids]
    test = phl7.iloc[test_ids]
    evaluate("Target only", training, test, n, repeat, seed, source_count=0)

print("Target-only model: 90 fits completed.")

## 3. Full source-data transfer

Pool the same target-enzyme training observations with **all 440** LCCICCG observations, fit a new GP, and evaluate exclusively on the paired held-out PES-H1 observations. This is pooled-data transfer, **not** fine-tuning a pretrained GP.

In [ ]:
phl7 = pd.read_excel(excel_file, sheet_name="PHL7", usecols=columns)
lcc = pd.read_excel(excel_file, sheet_name="LCC", usecols=columns)
assert len(lcc) == 440
assert np.isfinite(lcc[columns].to_numpy()).all()
assert not lcc.duplicated(features).any()

for n, repeat, seed, train_ids, test_ids in paired_splits(phl7):
    phl7_training = phl7.iloc[train_ids]
    test = phl7.iloc[test_ids]
    training = pd.concat([lcc, phl7_training], ignore_index=True)
    evaluate("Transfer full", training, test, n, repeat, seed, source_count=len(lcc))

print("Transfer full model: 90 fits completed, using 440 LCC observations each time.")

## 4. Response-range-calibrated transfer

For each split, select only LCCICCG observations whose measured BHET yield is within the **minimum–maximum yield interval of the PES-H1 training subset**. Fit a new GP on this selected source subset plus the paired target training subset. PES-H1 test responses are never used for source selection.

**Interpretation:** This is source-data filtering in *response space*, not explicit feature-space matching. The number of included source observations varies across repetitions.

In [ ]:
phl7 = pd.read_excel(excel_file, sheet_name="PHL7", usecols=columns)
lcc = pd.read_excel(excel_file, sheet_name="LCC", usecols=columns)

for n, repeat, seed, train_ids, test_ids in paired_splits(phl7):
    phl7_training = phl7.iloc[train_ids]
    test = phl7.iloc[test_ids]

    # Prepare the constrained LCC dataset from calibration observations only.
    minimum_yield = phl7_training[target].min()
    maximum_yield = phl7_training[target].max()
    lcc_calibrated = lcc.loc[lcc[target].between(minimum_yield, maximum_yield)]

    calibrated_counts.append({
        "PHL7 training n": n, "Repeat": repeat, "Seed": seed,
        "Minimum training yield": minimum_yield,
        "Maximum training yield": maximum_yield,
        "LCC training n": len(lcc_calibrated)
    })
    for row_id in lcc_calibrated.index:
        calibrated_membership.append({"PHL7 training n": n, "Repeat": repeat,
                                      "LCC Excel row": int(row_id) + 2})

    training = pd.concat([lcc_calibrated, phl7_training], ignore_index=True)
    evaluate("Transfer calibrated", training, test, n, repeat, seed,
             source_count=len(lcc_calibrated))

print("Transfer calibrated model: 90 fits completed.")

### 4.1. Source-data count audit

Report the actual numbers of selected LCCICCG observations per repetition. These **must not** be described as a fixed source size in Figure 7.

In [ ]:
calibrated_counts_df = pd.DataFrame(calibrated_counts)
count_summary = calibrated_counts_df.groupby("PHL7 training n")["LCC training n"].agg(
    Minimum="min", Median="median", Maximum="max"
)
display(count_summary)
with pd.option_context("display.max_rows", 100):
    display(calibrated_counts_df)

calibrated_counts_df.to_csv(output_folder / "calibrated_LCC_counts.csv", index=False)
pd.DataFrame(calibrated_membership).to_csv(
    output_folder / "calibrated_LCC_rows.csv", index=False
)

## 5. Illustrative parity plots

For visibility, plot measured versus predicted yield on the **same prespecified split** (10 target training observations; first repetition) for all three models. This example is not the basis for the summary statistics.

In [ ]:
model_names = ["Target only", "Transfer full", "Transfer calibrated"]
colors = ["#245c9c", "#ca6634", "#38916b"]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

for ax, model_name, color in zip(axes, model_names, colors):
    points = example_predictions[model_name]
    ax.scatter(points["Measured"], points["Predicted"], color=color)
    lower = min(points["Measured"].min(), points["Predicted"].min())
    upper = max(points["Measured"].max(), points["Predicted"].max())
    ax.plot([lower, upper], [lower, upper], "k--", linewidth=1)
    ax.set(title=model_name, xlabel="Measured", ylabel="Predicted")

fig.tight_layout()
fig.savefig(output_folder / "yield_parity.png", dpi=300, bbox_inches="tight")
fig.savefig(output_folder / "yield_parity.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

## 6. Held-out predictive performance

Summarize mean and standard deviation of $R^2$ across 30 paired random splits for each target training size. The SD quantifies variability across **overlapping random splits**, not a confidence interval. Negative $R^2$ values are retained.

In [ ]:
r2_by_split = pd.DataFrame(scores)
assert len(r2_by_split) == 270
assert np.isfinite(r2_by_split["Test R2"]).all()

r2_summary = r2_by_split.pivot_table(
    index="PHL7 training n", columns="Model", values="Test R2", aggfunc="mean"
).reindex(columns=model_names)
r2_summary.columns.name = "Mean held-out yield R2"

r2_sd = r2_by_split.pivot_table(
    index="PHL7 training n", columns="Model", values="Test R2", aggfunc="std"
).reindex(columns=model_names)
r2_sd.columns.name = "SD of held-out yield R2"

r2_by_split.to_csv(output_folder / "yield_R2_by_split.csv", index=False)
r2_summary.to_csv(output_folder / "yield_R2_mean.csv")
r2_sd.to_csv(output_folder / "yield_R2_sd.csv")
pd.DataFrame(fit_warnings, columns=["Model", "PHL7 training n", "Repeat", "Warning"]).to_csv(
    output_folder / "fit_warnings.csv", index=False
)
print(f"Optimizer warnings recorded: {len(fit_warnings)} (see fit_warnings.csv).")
display(r2_sd.round(3))
display(r2_summary.round(3))

## 7. Manuscript correspondence and reproducibility notes

- **Figure 7:** Use `yield_R2_mean.csv` and `yield_R2_sd.csv` for target-only, full-transfer, and calibrated-transfer results.
- **Source counts:** Use `calibrated_LCC_counts.csv` to report variable calibrated source sizes, rather than fixed totals of 230, 232, and 234.
- **Methodological limit:** Improved mean $R^2$ is descriptive; this notebook does not establish statistical significance or validate performance on a completely independent enzyme dataset.
- **Input requirements:** `PET_DegradationTrain_Test.xlsx`, sheets `LCC` and `PHL7`. Maintain the original row contents/order for exact split replication.
- **Execution:** Run all cells top-to-bottom in a fresh kernel; output CSVs and figures are written to `yield_r2_results/`.

The original manuscript labels PES-H1 as PHL7; the source notebook uses the worksheet name `PHL7`.